# Task 6: Partition Strategy Benchmark

This notebook compares two different Delta table storage designs for the
**taxi_trips** dataset (the raw fact table from Task 3).

**What is held constant (so the comparison is fair):**
- Same source rows, same schema, same Spark session, same machine.
- Both tables are written back-to-back in this single run, so JVM warm-up
  and OS-level effects apply roughly equally to both.

**What is different between the two designs:**
- **Strategy A - time-based:** partitioned by `pickup_year`, `pickup_month`
- **Strategy B - location-based:** partitioned by `pickup_borough`.

We measure, for each strategy:
1. Ingestion time (how long the `.write` takes)
2. Storage size on disk
3. Number of output files
4. Query latency for the three required queries:
   - number of trips per borough
   - average trip duration per day
   - average fare per borough

## 1. Setup

In [1]:
import os
import shutil
import time
from pathlib import Path

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

In [2]:
builder = (
    SparkSession.builder
    .appName("Task6-Partition-Benchmark")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/11 21:46:26 WARN Utils: Your hostname, Rex-Chang-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.113 instead (on interface en0)
26/09/11 21:46:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2ee6b926-ecf1-4dd2-bda9-1f5a7e4cf308;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io

Spark version: 4.2.0


In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a parent directory containing data/raw")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

BENCHMARK_DIR = PROCESSED_DIR / "benchmark"
STRATEGY_A_PATH = BENCHMARK_DIR / "taxi_trips_by_time"
STRATEGY_B_PATH = BENCHMARK_DIR / "taxi_trips_by_borough"

TAXI_TRIPS_PATH = PROCESSED_DIR / "taxi_trips"
TAXI_ZONES_PATH = PROCESSED_DIR / "taxi_zones"

for path in (TAXI_TRIPS_PATH, TAXI_ZONES_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing processed Delta table: {path}. Run ingestion_framework.ipynb first."
        )

print("Taxi trips source:", TAXI_TRIPS_PATH)
print("Strategy A (time) output:", STRATEGY_A_PATH)
print("Strategy B (borough) output:", STRATEGY_B_PATH)

Taxi trips source: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Strategy A (time) output: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/benchmark/taxi_trips_by_time
Strategy B (borough) output: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/benchmark/taxi_trips_by_borough


## 2. Build one shared source DataFrame

Both strategies must write the *exact same data*, so the only variable being
tested is the partitioning scheme. We add `pickup_borough` once here (by
joining the small `taxi_zones` lookup table) and reuse this single DataFrame
for both writes.

In [7]:
trips = spark.read.format("delta").load(str(TAXI_TRIPS_PATH))
zones = spark.read.format("delta").load(str(TAXI_ZONES_PATH))

pickup_zones = F.broadcast(
    zones.select(
        F.col("location_id").alias("pickup_zone_location_id"),
        F.col("borough").alias("pickup_borough"),
    )
)

benchmark_df = (
    trips
    .join(pickup_zones, trips.pickup_location_id == pickup_zones.pickup_zone_location_id, "left")
    .drop("pickup_zone_location_id")
    .withColumn("pickup_borough", F.coalesce(F.col("pickup_borough"), F.lit("Unknown")))
)
benchmark_df = benchmark_df.cache()
row_count = benchmark_df.count()
print("Rows to write for each strategy:", f"{row_count:,}")

zones.filter(F.col("borough") == "N/A").show()
zones.filter(F.col("borough") == "Unknown").show()

26/09/11 22:02:47 WARN CacheManager: Asked to cache already cached data.


Rows to write for each strategy: 9,417,689
+-----------+-------+--------------+------------+--------------------+--------------------+--------------------+---------------+
|location_id|borough|          zone|service_zone|        _source_file|   _ingestion_run_id|        _ingested_at|_schema_version|
+-----------+-------+--------------+------------+--------------------+--------------------+--------------------+---------------+
|        265|    N/A|Outside of NYC|         N/A|file:///Users/rex...|7aeb01a6-19f8-41b...|2026-09-10 05:36:...|          1.0.0|
+-----------+-------+--------------+------------+--------------------+--------------------+--------------------+---------------+

+-----------+-------+----+------------+--------------------+--------------------+--------------------+---------------+
|location_id|borough|zone|service_zone|        _source_file|   _ingestion_run_id|        _ingested_at|_schema_version|
+-----------+-------+----+------------+--------------------+-------------

26/09/12 00:34:23 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1048567 ms exceeds timeout 120000 ms
26/09/12 00:34:23 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/12 00:34:31 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:70

## 3. Helper functions

- `time_write(...)`: writes the DataFrame with a given partitioning and
  returns how long it took.
- `describe_storage(...)`: walks the output folder and reports total size
  and file count.
- `time_query(...)`: runs a query and returns both the result and how long
  it took to fully execute.

In [ ]:
def time_write(df, path: Path, partition_cols: list[str]) -> float:
    if path.exists():
        shutil.rmtree(path)  # start clean so old runs never leak into the size/file-count numbers
    started = time.perf_counter()
    (
        df
        # One file per partition value, instead of letting each of the 64
        .repartition(*[F.col(c) for c in partition_cols])
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy(*partition_cols)
        .save(str(path))
    )
    return time.perf_counter() - started


def describe_storage(path: Path) -> dict:
    total_bytes = 0
    file_count = 0
    for root, dirs, files in os.walk(path):
        dirs[:] = [d for d in dirs if d != "_delta_log"]
        for name in files:
            total_bytes += (Path(root) / name).stat().st_size
            file_count += 1
    return {"size_mb": total_bytes / (1024 ** 2), "file_count": file_count}


def time_query(df) -> tuple[list, float]:
    started = time.perf_counter()
    rows = df.collect()
    elapsed = time.perf_counter() - started
    return rows, elapsed

## 4. Write both strategies and measure ingestion + storage

In [6]:
results = {}

ingestion_time_a = time_write(benchmark_df, STRATEGY_A_PATH, ["pickup_year", "pickup_month"])
results["strategy_a_time_based"] = {
    "partition_columns": ["pickup_year", "pickup_month"],
    "ingestion_time_seconds": ingestion_time_a,
    **describe_storage(STRATEGY_A_PATH),
}

ingestion_time_b = time_write(benchmark_df, STRATEGY_B_PATH, ["pickup_borough"])
results["strategy_b_borough_based"] = {
    "partition_columns": ["pickup_borough"],
    "ingestion_time_seconds": ingestion_time_b,
    **describe_storage(STRATEGY_B_PATH),
}

# We no longer need the cached source now that both copies are written.
benchmark_df.unpersist()

for name, stats in results.items():
    print(name, stats)

strategy_a_time_based {'partition_columns': ['pickup_year', 'pickup_month'], 'ingestion_time_seconds': 14.73321595788002, 'size_mb': 842.7888679504395, 'file_count': 16}
strategy_b_borough_based {'partition_columns': ['pickup_borough'], 'ingestion_time_seconds': 26.597456249874085, 'size_mb': 845.7840309143066, 'file_count': 16}


## 5. Run the three required queries against each strategy

Each strategy is **re-read from disk** here rather than reused from the
cached `benchmark_df` above. We want to measure how fast a
real, cold query against each *stored table* is, not how fast a query
against an in-memory cache is.

In [ ]:
strategy_a = spark.read.format("delta").load(str(STRATEGY_A_PATH))
strategy_b = spark.read.format("delta").load(str(STRATEGY_B_PATH))

queries = {
    "trips_per_borough": lambda df: df.groupBy("pickup_borough").count(),
    "avg_trip_duration_per_day": lambda df: df.groupBy("pickup_date").agg(
        F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes")
    ),
    "avg_fare_per_borough": lambda df: df.groupBy("pickup_borough").agg(
        F.avg("fare_amount").alias("avg_fare_amount")
    ),
}

query_results = {}
for strategy_name, df in [("strategy_a_time_based", strategy_a), ("strategy_b_borough_based", strategy_b)]:
    for query_name, query_fn in queries.items():
        rows, elapsed = time_query(query_fn(df))
        query_results[(strategy_name, query_name)] = elapsed
        print(f"{strategy_name:25s} {query_name:28s} {elapsed:6.2f}s  ({len(rows)} result rows)")

strategy_a_time_based     trips_per_borough              0.41s  (8 result rows)
strategy_a_time_based     avg_trip_duration_per_day      0.27s  (96 result rows)
strategy_a_time_based     avg_fare_per_borough           0.23s  (8 result rows)
strategy_b_borough_based  trips_per_borough              0.30s  (8 result rows)
strategy_b_borough_based  avg_trip_duration_per_day      0.29s  (96 result rows)
strategy_b_borough_based  avg_fare_per_borough           0.20s  (8 result rows)


## 6. Combine everything into one comparison table

In [8]:
import pandas as pd

rows = []
for strategy_name, stats in results.items():
    row = {
        "strategy": strategy_name,
        "partition_columns": ", ".join(stats["partition_columns"]),
        "ingestion_time_s": round(stats["ingestion_time_seconds"], 2),
        "storage_size_mb": round(stats["size_mb"], 1),
        "file_count": stats["file_count"],
    }
    for query_name in queries:
        row[f"{query_name}_s"] = round(query_results[(strategy_name, query_name)], 2)
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df

,strategy,partition_columns,ingestion_time_s,storage_size_mb,file_count,trips_per_borough_s,avg_trip_duration_per_day_s,avg_fare_per_borough_s
0,strategy_a_time_based,"pickup_year, pickup_month",14.73,842.8,16,0.41,0.27,0.23
1,strategy_b_borough_based,pickup_borough,26.60,845.8,16,0.30,0.29,0.20


## 7. Repeat each query to check for warm-up / cache effects

Spark's first query against freshly-read data pays a one-time cost for
JVM code generation. We run each query a second time shows how much of
the timing above was warm-up versus the true steady-state cost.

In [ ]:
for strategy_name, df in [("strategy_a_time_based", strategy_a), ("strategy_b_borough_based", strategy_b)]:
    for query_name, query_fn in queries.items():
        _, elapsed = time_query(query_fn(df))
        print(f"[2nd run] {strategy_name:25s} {query_name:28s} {elapsed:6.2f}s")

[2nd run] strategy_a_time_based     trips_per_borough              0.20s
[2nd run] strategy_a_time_based     avg_trip_duration_per_day      0.14s
[2nd run] strategy_a_time_based     avg_fare_per_borough           0.19s
[2nd run] strategy_b_borough_based  trips_per_borough              0.13s
[2nd run] strategy_b_borough_based  avg_trip_duration_per_day      0.19s
[2nd run] strategy_b_borough_based  avg_fare_per_borough           0.15s


26/09/11 15:48:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 357058 ms exceeds timeout 120000 ms
26/09/11 15:48:00 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/11 15:48:04 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707